## Code Walkthrough
_Study Unit: ARI5118 - Deep Learning for Computer Vision_<br>
_Topic: CNN Feature Visualisation and DeepDream_<br>
_Author: David Farrugia_

This notebook is a practical walkthrough of **CNN feature visualisation** `using a pretrained VGG16 network`. Rather than using a single library function to generate outputs, the notebook builds a small pipeline that loads images, passes them through selected convolutional layers, captures intermediate activations and then uses gradient-based optimisation to produce visual outputs.

The main techniques demonstrated are **feature map extraction**, **activation maximisation** and **DeepDream**. By inspecting and modifying these internal representations, we can better understand what types of patterns the network responds to.

Each section in this notebook introduces and explains the reasoning behind the code, and highlights the effect of important parameters such as step size, regularisation, number of octaves and octave scale.

Lastly, this notebook was also used to generate the output images used in the accompanying interactive simulator. The simulator does not generate the visualisations in real time; instead, it displays these pre-generated outputs based on the selected method, layer and parameter settings.

**>> Link to simulatory: [https://davidf-22.github.io/ARI5118-DeepLearningCV_Project/](https://davidf-22.github.io/ARI5118-DeepLearningCV_Project/)**

## 0. Setup

The first step is to import the libraries used throughout the notebook.

- `torch` and `torchvision` are used to load the pretrained CNN and compute gradients.
- `PIL` is used for image loading and RGB conversion.
- `matplotlib` is used to save visual outputs.
- `pathlib` is used to handle file paths cleanly.

The pipeline uses PyTorch and torchvision to load a pretrained VGG16 model, PIL to read image files, pathlib to manage paths cleanly, and matplotlib to save the generated visualisations.

A fixed random seed is also set so that the experiment is more reproducible. This matters because some parts of the pipeline, especially DeepDream, introduce small amounts of noise. Without a fixed seed, the generated images may look slightly different each time the notebook is executed.

In [1]:
import time
import random
from pathlib import Path

import torch
from torchvision import models, transforms
from PIL import Image

import matplotlib.pyplot as plt

SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)

### 0.1. Selecting the computation device

If a GPU is available, PyTorch will use it. Otherwise, the pipeline will run on the CPU. This is useful because the notebook remains executable on other machines, even if generation is slower without a GPU.

In [2]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f">> Using device: {DEVICE}")

>> Using device: cuda


In [3]:
start_time = time.time()

## 1. Configuration

This section centralises the main settings used throughout the notebook. All major parameters are grouped into a single `CONFIG` dictionary, making the experiment easier to modify because important values can be changed from one place rather than being scattered across the code.

The configuration controls:

- where input and output images are stored;
- the image size used by the model;
- how many feature-map channels are saved;
- activation maximisation settings such as step size and number of optimisation steps;
- DeepDream settings such as octaves, octave scale, noise, and step size;
- Boolean flags that allow parts of the notebook to be skipped during testing.

The parameter lists, such as `ACT_MAX_STEP_SIZE`, `DEEPDREAM_OCTAVES`, and `DEEPDREAM_OCTAVE_SCALE`, are used later to generate multiple outputs. This is useful because the notebook is not only meant to produce one result, but also to show how changing key parameters affects the final visualisation.

In [ ]:
CONFIG = {
    # DIRECTORIES
    "input_imgs_dir": Path("./simulator/assets/imgs/input_imgs"),
    "output_dir": Path("./simulator/assets/imgs/output_imgs"),

    # IMAGE SETTINGS
    "IMAGE_SIZE": 224,

    # FEATURE MAPS
    "NUM_FILTERS": 10,

    # ACTIVATION MAXIMISATION
    "ACT_MAX_STEPS": 100,
    "ACT_MAX_STEP_SIZE": [0.001, 0.01, 0.1],

    # REGULARISATION
    "L2_LAMBDA": 1e-5,

    # DEEPDREAM
    "DEEPDREAM_STEPS": 40,
    "DEEPDREAM_NOISE": 0.01,
    "DEEPDREAM_STEP_SIZE": 0.001,
    "DEEPDREAM_OCTAVES": [2, 3, 4],
    "DEEPDREAM_OCTAVE_SCALE": [0.6, 0.8, 1.0, 1.2, 1.4],
    
    # BOOL FLAGS - Used for faster debugging and testing and incase of Out of Memory Errors
    "GET_FEATURE_MAPS": True,
    "RUN_ACT_MAX": True,
    "RUN_DEEPDREAM": True
}

### 1.1. Validating Directories

Before running the model, the notebook checks that the input image folder exists. If the folder is missing, the notebook stops.
The output folder is created automatically if it does not already exist. This keeps the generated images organised and avoids manual setup before every run.

In [5]:
INPUT_DIR = CONFIG["input_imgs_dir"]
OUTPUT_DIR = CONFIG["output_dir"]

if not INPUT_DIR.exists():
    raise FileNotFoundError(f">> [ERROR] Input image folder not found: {INPUT_DIR.resolve()}")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

## 2. Loading Input Images Paths

This stage collects all supported input images from the input directory. The notebook accepts `.jpg`, `.jpeg`, and `.png` files.

Only the image paths are collected here. The actual image loading is done later inside the feature-map, activation-maximisation and DeepDream functions. This avoids keeping unnecessary image tensors in memory.

The printed output acts as a quick sanity check: it confirms how many images were found and which files will be processed.

In [6]:
IMAGE_PATHS = []

# Load all images from the directory
image_extensions = ['*.jpg', '*.jpeg', '*.png']
images = []

for ext in image_extensions:
    IMAGE_PATHS.extend(INPUT_DIR.glob(ext))

if len(IMAGE_PATHS) == 0:
    raise ValueError(f">> [ERROR] No images found in: {INPUT_DIR.resolve()}")

print(f">> --- [Found {len(IMAGE_PATHS)} input images] ---:")
for path in IMAGE_PATHS:
    print(f">> {path.name}")

>> --- [Found 5 input images] ---:
>> Black-Cat.png
>> Golden-Retriever.png
>> mountains.png
>> Parrot.png
>> Tokyo_Shibuya.png


## 3. Load pretrained model

This walkthrough uses a pretrained VGG16 model. VGG16 is useful for feature visualisation because its architecture is simple and sequential, making it easier to select specific convolutional layers and interpret their outputs.

The model is placed in evaluation mode using `model.eval()`. This is important because it disables training-specific behaviour and makes the model behave consistently during visualisation. Additionally, in this notebook, the network weights are not updated; only the input image is analysed or optimised.

In [7]:
weights = models.VGG16_Weights.DEFAULT
model = models.vgg16(weights=weights).to(DEVICE)
model.eval()

VGG(
  (features): Sequential(
    (0): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU(inplace=True)
    (2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (3): ReLU(inplace=True)
    (4): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (5): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (6): ReLU(inplace=True)
    (7): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (8): ReLU(inplace=True)
    (9): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (10): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (11): ReLU(inplace=True)
    (12): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (13): ReLU(inplace=True)
    (14): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (15): ReLU(inplace=True)
    (16): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1

### 3.1 Select Convolution Layers

Instead of inspecting every layer in VGG16, the notebook selects five early convolutional layers. These layers are labelled from `conv1` to `conv5` for readability.

This selection allows the notebook to demonstrate the idea of a feature hierarchy. Earlier layers usually respond to simpler visual patterns such as edges, colours, and textures. Deeper layers tend to represent more complex combinations of those patterns. By saving outputs from multiple layers, the notebook can show how the network representation changes with depth.

In [8]:
# First 5 convolutional layers
TARGET_LAYERS = {
    "conv1": model.features[0],
    "conv2": model.features[2],
    "conv3": model.features[5],
    "conv4": model.features[7],
    "conv5": model.features[10],
}

## 4. Image Transforms and Helper Functions

The model expects images to be prepared in a specific way before they are passed through VGG16. This section defines the transforms and helper functions used by the rest of the notebook.

There are two different transform pipelines:

- `preprocess` prepares images for VGG16 using ImageNet normalisation.
- `plain_transform` keeps images in the `[0, 1]` range, which is useful when directly modifying pixels during activation maximisation and DeepDream.
<br><br>
> **PyTorch Documentation: [https://docs.pytorch.org/vision/0.12/models](https://docs.pytorch.org/vision/0.12/models)**

### 4.1. Image Transforms

The `preprocess` transform resizes the image, converts it to a tensor, and normalises it using the ImageNet mean and standard deviation. This is the correct input format for pretrained VGG16.

The `plain_transform` also resizes and converts the image to a tensor, but it does not normalise the values. This makes it easier to optimise and clamp the image during gradient-based visualisation.

In [9]:
preprocess = transforms.Compose([
    transforms.Resize((CONFIG["IMAGE_SIZE"], CONFIG["IMAGE_SIZE"])),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

plain_transform = transforms.Compose([
    transforms.Resize((CONFIG["IMAGE_SIZE"], CONFIG["IMAGE_SIZE"])),
    transforms.ToTensor()
])

### 4.2. Helper Functions

This helper opens an image from disk and converts it to RGB. RGB conversion is important because VGG16 expects three-channel colour images.

In [10]:
def load_image(path):
    return Image.open(path).convert("RGB")

### 4.3. Converting tensors back to displayable images

Model-ready tensors are not always directly displayable because they may be normalised or arranged in channel-first format `[C, H, W]`.

This helper converts tensors back into a format suitable for saving or displaying:

- it moves the tensor back to the CPU;
- it approximately reverses ImageNet normalisation when needed;
- it clamps values to the `[0, 1]` range;
- it rearranges the tensor into image format `[H, W, C]`.

In [ ]:
def tensor_to_display_img(tensor):
    """
    Convert a PyTorch image tensor into a NumPy image that can be displayed
    using matplotlib

    Expected input shape:
        [3, H, W]

    Output shape:
        [H, W, 3]
    """
    # Detach from gradients, move to CPU, and avoid changing the original tensor
    tensor = tensor.detach().cpu().clone()

    # Undo ImageNet normalization approximately if needed
    if tensor.min() < 0:
        mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
        std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
        tensor = tensor * std + mean

    # Clamp values to the valid image display range
    tensor = tensor.clamp(0, 1)
    
    return tensor.permute(1, 2, 0).numpy()

### 4.4. Saving generated images

This helper saves an RGB tensor as an image file. It is used for activation maximisation and DeepDream outputs.
The output folder is created automatically, which keeps the notebook robust even when new parameter combinations create new folder paths.

In [ ]:
def save_tensor_image(tensor, save_path):
    save_path.parent.mkdir(parents=True, exist_ok=True)

    img = tensor_to_display_img(tensor)

    plt.figure(figsize=(4, 4))
    plt.imshow(img)
    plt.axis("off")
    plt.tight_layout()
    plt.savefig(save_path, bbox_inches="tight", pad_inches=0)
    plt.close()


### 4.5. Saving individual feature maps

A feature map is a single channel from a convolutional layer activation. Since raw activation values may have different ranges, the feature map is normalised before saving.

A colour map is used to make stronger and weaker activations easier to distinguish visually. Brighter regions indicate areas where that filter responded more strongly.

In [ ]:
def save_feature_map(feature_map, save_path):
    # Create the directory if it doesn't exist
    save_path.parent.mkdir(parents=True, exist_ok=True)

    feature_map = feature_map.detach().cpu()                # Move to CPU and detach from gradients
    feature_map = feature_map - feature_map.min()           # Shift the values to make minimum 0
    feature_map = feature_map / (feature_map.max() + 1e-8)  # Normalize to [0, 1] range for better visualization

    plt.figure(figsize=(4, 4))
    plt.imshow(feature_map.numpy(), cmap="viridis")
    plt.axis("off")
    plt.tight_layout()
    plt.savefig(save_path, bbox_inches="tight", pad_inches=0)
    plt.close()

## 5. Register activation hooks


To inspect what happens inside the CNN, the notebook uses forward hooks. A forward hook is a small function that runs automatically when a selected layer processes an input image.

In this implementation, each hook stores the output activation of its layer in the `activations` dictionary. This avoids rewriting the model and allows intermediate layer outputs to be captured during a normal forward pass.

In [ ]:
activations = {}    # To store activations

def get_activation(name):
    # Create a hook function that remembers this specific layer name
    def hook(model, input, output):
        activations[name] = output
        
    # Return the custom hook so PyTorch can attach it to the layer
    return hook

hooks = []   # Store hook references

for name, layer in TARGET_LAYERS.items():
    # Attach a forward hook that records this layer's output
    hook = layer.register_forward_hook(get_activation(name))
    hooks.append(hook)

print(">> Hooks registered for:", list(TARGET_LAYERS.keys()))

>> Hooks registered for: ['conv1', 'conv2', 'conv3', 'conv4', 'conv5']


## 6. Generate feature maps

Feature maps show how each selected convolutional filter responds to a real input image. For every input image, the notebook:

1. loads and preprocesses the image;
2. performs a forward pass through VGG16;
3. reads the stored activations from the hooks;
4. saves the first `NUM_FILTERS` channels from each selected layer.

This gives a direct view of the feature hierarchy. Early layers usually preserve more spatial detail, while deeper layers tend to become more abstract and selective.

In [ ]:
if CONFIG["GET_FEATURE_MAPS"]:
    
    def generate_feature_maps():
        print("\n>> --- [Generating feature maps...] --- :")

        for image_path in IMAGE_PATHS:
            # Use the image filename to organise saved outputs
            image_name = image_path.stem

            image = load_image(image_path)
            input_tensor = preprocess(image).unsqueeze(0).to(DEVICE)

            # Run a forward pass so the registered hooks capture activations.
            with torch.no_grad():
                _ = model(input_tensor)

            for layer_name, activation in activations.items():
                activation = activation[0]  # Remove the batch dimension: [1, C, H, W] -> [C, H, W]

                # Limit how many channels are saved from this layer
                num_channels = min(CONFIG["NUM_FILTERS"], activation.shape[0])

                for channel_idx in range(num_channels):
                    feature_map = activation[channel_idx]   # Extract the specific channel's feature map
            
                    save_path = (OUTPUT_DIR / "feature_maps" / image_name / layer_name / f"filter{channel_idx + 1}.png")
                    save_feature_map(feature_map, save_path)

            print(f">> [OK] Saved feature maps for: {image_name}")
        print(">> [DONE] Feature map generation complete.")

    generate_feature_maps()
else:
    print(">> [SKIP] Feature map generation skipped due to config flag.")


>> --- [Generating feature maps...] --- :
>> [OK] Saved feature maps for: Black-Cat
>> [OK] Saved feature maps for: Golden-Retriever
>> [OK] Saved feature maps for: mountains
>> [OK] Saved feature maps for: Parrot
>> [OK] Saved feature maps for: Tokyo_Shibuya
>> [DONE] Feature map generation complete.


## 7. Activation Maximisation

Activation maximisation answers the question:

> _What kind of image would make a chosen filter respond strongly?_

In this notebook, the optimisation starts from a real input image rather than pure random noise. The image tensor is treated as the optimisable variable. During each step, the model calculates how the selected filter activation changes with respect to the input pixels, and the image is updated to increase that activation.

This section also allows comparison between different step sizes and between outputs with and without L2 regularisation.

**Key parameters to observe**

- **Step size:** controls how strongly the image is changed at each optimisation step. Small values produce subtle changes, while large values can create stronger but noisier patterns.
- **Number of steps:** controls how long the optimisation runs.
- **L2 regularisation:** discourages extreme pixel values and can make outputs smoother or less chaotic.
- **Layer and filter:** determine which internal feature is being maximised.

In [ ]:
if CONFIG["RUN_ACT_MAX"]:

    def activation_maximisation(image_path, layer_name, filter_idx, step_size, use_l2=False):
        if layer_name not in TARGET_LAYERS:
            raise ValueError(f"Unknown layer: {layer_name}")

        image = load_image(image_path)

        # Keep values in [0, 1] because the image itself will be optimised
        input_img = plain_transform(image).unsqueeze(0).to(DEVICE)
        input_img = input_img.clone()

        # Enable gradients for the input image instead of the model weights
        input_img.requires_grad_(True)

        # Adam updates the image pixels directly
        optimizer = torch.optim.Adam(
            [input_img],
            lr=step_size
        )

        for step in range(CONFIG["ACT_MAX_STEPS"]):
            optimizer.zero_grad()   # Clear previous gradients from the last step
            activations.clear()     # Adam updates the image pixels directly
            _ = model(input_img)    # Run a forward pass to compute activations and trigger hooks

            # Maximise the average response of the selected filter.
            target_activation = activations[layer_name][0, filter_idx].mean()
            loss = -target_activation

            if use_l2:
                loss = loss + CONFIG["L2_LAMBDA"] * torch.mean(input_img ** 2)

            loss.backward()
            optimizer.step()

            # Clamp after each update to keep the image displayable.
            with torch.no_grad():
                input_img.clamp_(0, 1)

        return input_img[0].detach()

### 7.1. Generating activation maximisation outputs

The generation function runs activation maximisation across all selected images, layers, filters, step sizes, and regularisation settings.

The outputs are saved using a folder structure that records the experiment settings. This is important for the simulator because each generated image can later be retrieved based on the selected method, layer, filter, step size, and regularisation option.

When reviewing the outputs, compare:

- the same filter across shallow and deep layers;
- the same filter with different step sizes;
- regularised and non-regularised versions of the same output.

In [ ]:
if CONFIG["RUN_ACT_MAX"]:
    
    def generate_activation_maximisation_outputs():
        print("\n>> --- [Generating activation maximisation outputs...] --- :")
        
        for image_path in IMAGE_PATHS:
            image_name = image_path.stem
            
            # Test multiple optimisation strengths for comparison
            for step_size in CONFIG["ACT_MAX_STEP_SIZE"]:
                for use_l2 in [False, True]:

                    reg_folder = ("l2_regularisation" if use_l2 else "no_regularisation")

                    for layer_name in TARGET_LAYERS.keys():
                        for filter_idx in range(CONFIG["NUM_FILTERS"]):

                            img = activation_maximisation(image_path, layer_name, filter_idx, step_size, use_l2)

                            # Organise outputs by regularisation, image, step size, layer, and filter.
                            save_path = (OUTPUT_DIR / "activation_maximisation" / reg_folder / image_name / f"step_{step_size}" 
                                        / layer_name / f"filter{filter_idx + 1}.png")
                            save_tensor_image(img, save_path)

                        print(f">> [OK] Saved Activation Maximisation | {image_name} | {layer_name} | step={step_size} | L2={use_l2}")
            print()
        print(">> [DONE] Activation maximisation complete.")
    
    generate_activation_maximisation_outputs()
else:
    print(">> [SKIP] Activation maximisation skipped due to config flag.")


>> --- [Generating activation maximisation outputs...] --- :
>> [OK] Saved Activation Maximisation | Black-Cat | conv1 | step=0.001 | L2=False
>> [OK] Saved Activation Maximisation | Black-Cat | conv2 | step=0.001 | L2=False
>> [OK] Saved Activation Maximisation | Black-Cat | conv3 | step=0.001 | L2=False
>> [OK] Saved Activation Maximisation | Black-Cat | conv4 | step=0.001 | L2=False
>> [OK] Saved Activation Maximisation | Black-Cat | conv5 | step=0.001 | L2=False
>> [OK] Saved Activation Maximisation | Black-Cat | conv1 | step=0.001 | L2=True
>> [OK] Saved Activation Maximisation | Black-Cat | conv2 | step=0.001 | L2=True
>> [OK] Saved Activation Maximisation | Black-Cat | conv3 | step=0.001 | L2=True
>> [OK] Saved Activation Maximisation | Black-Cat | conv4 | step=0.001 | L2=True
>> [OK] Saved Activation Maximisation | Black-Cat | conv5 | step=0.001 | L2=True
>> [OK] Saved Activation Maximisation | Black-Cat | conv1 | step=0.01 | L2=False
>> [OK] Saved Activation Maximisation | Bl

## 8. DeepDream

DeepDream is another gradient-based visualisation technique. Instead of creating an image that maximises one specific filter, it amplifies patterns that the CNN already detects in an existing image.

The idea is:

1. pass the image through the CNN;
2. choose a target layer;
3. maximise the average activation of that layer;
4. update the input image using the gradient;
5. repeat the process so detected patterns become stronger.

This often produces dream-like effects because the network reinforces patterns it already recognises. Lower layers tend to amplify textures, edges, and colour patterns, while deeper layers may produce more abstract or repeated structures.

The implementation also uses octaves. This means the image is processed at multiple scales. Starting at smaller scales and gradually increasing the resolution helps DeepDream create patterns that appear across different levels of detail, rather than only affecting small local textures.


### 8.1. One DeepDream optimisation pass

The `deepdream_step` function performs the core gradient update. It computes the mean activation of a selected layer and adjusts the image to increase that activation.

The gradient is normalised before being applied. This helps prevent the update from becoming too unstable when gradients become very large or very small

In [ ]:
if CONFIG["RUN_DEEPDREAM"]:

    def deepdream_step(input_img, layer_name):
        input_img.requires_grad_(True)  # Enable gradients for the input image instead of the model weights

        for step in range(CONFIG["DEEPDREAM_STEPS"]):
            model.zero_grad()
            activations.clear()
            _ = model(input_img)

            # Maximise the average activation of the selected layer
            loss = activations[layer_name].mean()
            loss.backward()

            with torch.no_grad():
                grad = input_img.grad

                grad = grad / (grad.std() + 1e-8) # Normalize the gradient to prevent exploding updates

                # Move the image in the direction of the gradient to enhance features that activate the layer
                input_img += CONFIG["DEEPDREAM_STEP_SIZE"] * grad
                input_img.clamp_(0, 1)
                
                # Clear before next step
                input_img.grad.zero_()

        return input_img.detach()

### 8.2. Resizing tensors for octave processing

DeepDream often uses octaves, which means the image is processed at multiple scales. This helper resizes an image tensor while keeping it in tensor format. Processing at different scales helps the visual effect appear at both small texture levels and larger structural levels.

In [ ]:
if CONFIG["RUN_DEEPDREAM"]:

    def resize_tensor(tensor, size):
        return torch.nn.functional.interpolate(tensor, size=size, mode="bilinear",align_corners=False)

### 8.3. Multi-octave DeepDream

The `deepdream` function applies the DeepDream update across multiple image sizes.

The image first receives a small amount of noise so the optimisation has more variation to amplify. The image is then processed from smaller to larger octave sizes. This allows patterns found at lower resolutions to influence the final high-resolution output.

**Key parameters to observe**

- **Octaves:** increasing the number of octaves usually strengthens the dream-like effect across multiple scales.
- **Octave scale:** controls how aggressively the image is resized between octaves.
- **DeepDream step size:** controls how strongly each gradient update changes the image.
- **Target layer:** shallow layers tend to amplify textures and edges, while deeper layers can produce more complex patterns.

In [ ]:
if CONFIG["RUN_DEEPDREAM"]:
    
    def deepdream(image_path, layer_name, octaves, octave_scale):
        image = load_image(image_path)
        img_tensor = plain_transform(image).unsqueeze(0).to(DEVICE)

        original_size = CONFIG["IMAGE_SIZE"]

        octave_sizes = []

        size = original_size
        for _ in range(octaves):
            octave_sizes.append(int(size))
            size = size / octave_scale

        # Process from small to large so coarse patterns influence later detail.
        octave_sizes = list(reversed(octave_sizes))

        dream = img_tensor
        
        # Add slight noise to give DeepDream more variation to amplify.
        dream += CONFIG["DEEPDREAM_NOISE"] * torch.randn_like(dream)
        dream = dream.clamp(0, 1)

        for octave_size in octave_sizes:
            # Resize before each octave-based optimisation pass.
            dream = resize_tensor(dream, (octave_size, octave_size))
            dream = deepdream_step(dream, layer_name)

        # Return the final dream image at the original configured size.
        dream = resize_tensor(dream, (CONFIG["IMAGE_SIZE"], CONFIG["IMAGE_SIZE"]))

        return dream[0].detach()

### 8.4. Generating DeepDream outputs

This function generates DeepDream images for every input image, selected layer, octave count, and octave scale.

The saved folder structure records the parameters used for each output. This makes it possible to compare results directly in the simulator, especially when changing octave and scale values.

When reviewing the generated images, look for how the effect changes as:

- the target layer becomes deeper;
- the number of octaves increases;
- the octave scale changes.

In [ ]:
if CONFIG["RUN_DEEPDREAM"]:
    
    def generate_deepdream_outputs():
        print("\n>> --- [Generating DeepDream outputs...] --- :")

        for image_path in IMAGE_PATHS:
            image_name = image_path.stem
            
            # Test multiple octave counts and scaling values for comparison.
            for octaves in CONFIG["DEEPDREAM_OCTAVES"]:
                for scale in CONFIG["DEEPDREAM_OCTAVE_SCALE"]:

                    for layer_name in TARGET_LAYERS.keys():
                        dream_img = deepdream(image_path, layer_name, octaves, scale)

                         # Organise outputs by image, octave setting, scale, and layer.
                        save_path = (OUTPUT_DIR / "deepdream" / image_name / f"octaves_{octaves}" / f"scale_{scale}" / layer_name / "dream.png")
                        save_tensor_image(dream_img, save_path)

                    print(f">> [OK] Saved {image_name} | oct={octaves} | scale={scale}")
            print()
        print(">> [DONE] DeepDream generation complete.")

    generate_deepdream_outputs()
else:
    print(">> [SKIP] DeepDream generation skipped due to config flag.")


>> --- [Generating DeepDream outputs...] --- :
>> [OK] Saved Black-Cat | oct=2 | scale=0.6
>> [OK] Saved Black-Cat | oct=2 | scale=0.8
>> [OK] Saved Black-Cat | oct=2 | scale=1.0
>> [OK] Saved Black-Cat | oct=2 | scale=1.2
>> [OK] Saved Black-Cat | oct=2 | scale=1.4
>> [OK] Saved Black-Cat | oct=3 | scale=0.6
>> [OK] Saved Black-Cat | oct=3 | scale=0.8
>> [OK] Saved Black-Cat | oct=3 | scale=1.0
>> [OK] Saved Black-Cat | oct=3 | scale=1.2
>> [OK] Saved Black-Cat | oct=3 | scale=1.4
>> [OK] Saved Black-Cat | oct=4 | scale=0.6
>> [OK] Saved Black-Cat | oct=4 | scale=0.8
>> [OK] Saved Black-Cat | oct=4 | scale=1.0
>> [OK] Saved Black-Cat | oct=4 | scale=1.2
>> [OK] Saved Black-Cat | oct=4 | scale=1.4

>> [OK] Saved Golden-Retriever | oct=2 | scale=0.6
>> [OK] Saved Golden-Retriever | oct=2 | scale=0.8
>> [OK] Saved Golden-Retriever | oct=2 | scale=1.0
>> [OK] Saved Golden-Retriever | oct=2 | scale=1.2
>> [OK] Saved Golden-Retriever | oct=2 | scale=1.4
>> [OK] Saved Golden-Retriever | oct

## 9. Clean up hooks and timing

After all visualisations are generated, the hooks are removed from the model. This is good practice because hooks remain attached until they are explicitly removed.

The notebook then prints the elapsed runtime in seconds and minutes. This makes it easier to compare how changes to the configuration affect total execution time.

In [22]:
for hook in hooks:
    hook.remove()

In [23]:
elapsed_time = time.time() - start_time
print(f">> Elapsed time: {elapsed_time:.3f} seconds | {(elapsed_time / 60):.2f} minutes")

>> Elapsed time: 2296.103 seconds | 38.27 minutes
